# ThaiSum Training Notebook (Colab)

This notebook is a step-by-step walkthrough for Thai summarization fine-tuning with **ThaiSum + Qwen3-4B + LoRA + LLaMA Factory**.

It is designed for learning first:
- load dataset
- visualize dataset
- prepare training files
- train with clear config values
- save adapter and merged model
- evaluate with ROUGE
- run inference on validation data and custom text

Repo mapping:
- Dataset prep logic: `prepare_thaisum_dataset.py`
- Dataset registry: `script/dataset_info.config.json`
- Training config: `script/yaml/2_qwen3_lora_sft_thaisum.config.yaml`
- Merge config: `script/yaml/3_merge_lora_qwen3_thaisum.config.yaml`
- Evaluation flow: `evaluate_summarization.py`
- Inference flow: `summarization_inference.py`

## 1. Install Dependencies

This section installs the packages needed for Colab. The versions are intentionally simple and easy to understand.

In [ ]:
!pip install -q datasets transformers accelerate peft sentencepiece rouge-score pandas matplotlib seaborn
!pip install -q llamafactory

## 2. Runtime Check

In [ ]:
import os
import json
import re
from pathlib import Path

import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!nvidia-smi

## 3. Set Paths and Default Config

These defaults follow the current repository configuration.

In [ ]:
WORKDIR = '/content/llm_finetune_thaisum'
DATA_DIR = f'{WORKDIR}/dataset'
RAW_DATA_DIR = f'{DATA_DIR}/thaisum_hf'
FORMAT_DATA_DIR = f'{DATA_DIR}/format_dataset/thai_news_summary'
TOKENIZED_DIR = f'{DATA_DIR}/tokenized_dataset/thai_news_summary'
SCRIPT_DIR = f'{WORKDIR}/script'
YAML_DIR = f'{SCRIPT_DIR}/yaml'
OUTPUT_DIR = f'{WORKDIR}/trained_model'

MODEL_NAME = 'Qwen/Qwen3-4B'
DATASET_NAME = 'pythainlp/thaisum'
DATASET_ALIAS = 'thai_news_summary'

CUTOFF_LEN = 4096
MAX_SAMPLES = 50000
LORA_RANK = 16
LORA_ALPHA = 32
LORA_TARGET = 'all'
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-5
NUM_TRAIN_EPOCHS = 3
WARMUP_RATIO = 0.1
LR_SCHEDULER_TYPE = 'cosine'
MAX_NEW_TOKENS = 256
EVAL_LIMIT = 100

for path in [WORKDIR, DATA_DIR, RAW_DATA_DIR, FORMAT_DATA_DIR, TOKENIZED_DIR, SCRIPT_DIR, YAML_DIR, OUTPUT_DIR]:
    Path(path).mkdir(parents=True, exist_ok=True)

print('Working directory:', WORKDIR)

## 4. Load ThaiSum Dataset

Repo mapping: same source dataset used by `prepare_thaisum_dataset.py`.

In [ ]:
dataset = load_dataset(DATASET_NAME)
dataset

In [ ]:
dataset['train'][0]

## 5. Dataset Visualization

This section helps you understand the data before training.

In [ ]:
split_sizes = {split: len(ds) for split, ds in dataset.items()}
pd.DataFrame({'split': split_sizes.keys(), 'rows': split_sizes.values()})

In [ ]:
train_df_preview = dataset['train'].select(range(5)).to_pandas()
train_df_preview[['title', 'body', 'summary']]

In [ ]:
length_df = dataset['train'].select(range(min(2000, len(dataset['train'])))).to_pandas()[['body', 'summary']].copy()
length_df['body_len'] = length_df['body'].fillna('').astype(str).str.len()
length_df['summary_len'] = length_df['summary'].fillna('').astype(str).str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(length_df['body_len'], bins=40, ax=axes[0])
axes[0].set_title('Body length distribution')
sns.histplot(length_df['summary_len'], bins=40, ax=axes[1])
axes[1].set_title('Summary length distribution')
plt.tight_layout()
plt.show()

In [ ]:
empty_check = []
for split_name, split_dataset in dataset.items():
    df = split_dataset.to_pandas()[['body', 'summary']].copy()
    df['body_empty'] = df['body'].fillna('').astype(str).str.strip().eq('')
    df['summary_empty'] = df['summary'].fillna('').astype(str).str.strip().eq('')
    empty_check.append({
        'split': split_name,
        'body_empty': int(df['body_empty'].sum()),
        'summary_empty': int(df['summary_empty'].sum())
    })

pd.DataFrame(empty_check)

## 6. Clean the Dataset

We keep only `body` and `summary`, then remove empty rows. This follows the repo dataset-preparation script.

In [ ]:
def is_valid_record(example):
    body = str(example.get('body') or '').strip()
    summary = str(example.get('summary') or '').strip()
    return bool(body and summary)

keep_columns = {'body', 'summary'}
drop_columns = [col for col in dataset['train'].column_names if col not in keep_columns]
clean_dataset = dataset.remove_columns(drop_columns)
clean_dataset = clean_dataset.filter(is_valid_record)
clean_dataset

## 7. Save Cleaned Raw Splits

Repo mapping: same output idea as `dataset/thaisum_hf/*.jsonl`.

In [ ]:
clean_dataset.save_to_disk(RAW_DATA_DIR)

for split_name, split_dataset in clean_dataset.items():
    output_path = f'{RAW_DATA_DIR}/{split_name}.jsonl'
    split_dataset.to_json(output_path, force_ascii=False)
    print(split_name, '->', output_path)

## 8. Convert to ShareGPT Format for LLaMA Factory

Repo mapping: this matches the `thai_news_summary` dataset entry in `script/dataset_info.config.json`.

In [ ]:
SYSTEM_PROMPT = (
    'คุณเป็นผู้ช่วยสรุปข่าวภาษาไทย\n'
    'สรุปเฉพาะจากข้อมูลในบทความต้นฉบับเท่านั้น\n'
    'เขียนสรุปให้กระชับ ชัดเจน และไม่เพิ่มข้อเท็จจริงที่ไม่มีในต้นฉบับ'
)

def to_sharegpt_rows(split_dataset):
    rows = []
    for row in split_dataset:
        rows.append({
            'messages': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': str(row['body']).strip()},
                {'role': 'assistant', 'content': str(row['summary']).strip()},
            ]
        })
    return rows

In [ ]:
for split_name, split_dataset in clean_dataset.items():
    sharegpt_rows = to_sharegpt_rows(split_dataset)
    output_path = f'{FORMAT_DATA_DIR}/{DATASET_ALIAS}_{split_name}.json'
    with open(output_path, 'w', encoding='utf-8') as f:
        for row in sharegpt_rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(split_name, '->', output_path)

In [ ]:
pd.read_json(f'{FORMAT_DATA_DIR}/{DATASET_ALIAS}_train.json', lines=True).head(2)

## 9. Write Dataset Registry and YAML Configs

These are notebook-generated versions of the same files used in the repo.

Config meaning:
- `cutoff_len=4096`: max sequence length
- `lora_rank=16`: LoRA capacity
- `lora_alpha=32`: LoRA scaling
- `per_device_train_batch_size=4`: micro-batch size per GPU
- `gradient_accumulation_steps=4`: effective batch growth without larger GPU memory
- `learning_rate=2e-5`: training step size
- `num_train_epochs=3`: full passes over training data
- `warmup_ratio=0.1`: warmup part of training
- `lr_scheduler_type=cosine`: learning rate decay style

In [ ]:
dataset_info = {
    DATASET_ALIAS: {
        'file_name': f'{FORMAT_DATA_DIR}/{DATASET_ALIAS}_train.json',
        'formatting': 'sharegpt',
        'columns': {'messages': 'messages'},
        'tags': {
            'role_tag': 'role',
            'content_tag': 'content',
            'user_tag': 'user',
            'assistant_tag': 'assistant',
            'system_tag': 'system'
        }
    }
}

with open(f'{SCRIPT_DIR}/dataset_info.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(json.dumps(dataset_info, ensure_ascii=False, indent=2))

In [ ]:
preprocess_yaml = f"""### model
model_name_or_path: {MODEL_NAME}

dataset: {DATASET_ALIAS}
template: qwen3
cutoff_len: {CUTOFF_LEN}
overwrite_cache: true
preprocessing_num_workers: 2
dataset_dir: {SCRIPT_DIR}
tokenized_path: {TOKENIZED_DIR}

### output
output_dir: {TOKENIZED_DIR}
overwrite_output_dir: true
"""

train_yaml = f"""### model
model_name_or_path: {MODEL_NAME}

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: {LORA_RANK}
lora_alpha: {LORA_ALPHA}
lora_target: {LORA_TARGET}

### dataset
dataset: {DATASET_ALIAS}
template: qwen3
cutoff_len: {CUTOFF_LEN}
max_samples: {MAX_SAMPLES}
overwrite_cache: true
preprocessing_num_workers: 2
dataloader_num_workers: 1
dataset_dir: {SCRIPT_DIR}
tokenized_path: {TOKENIZED_DIR}

### output
output_dir: {OUTPUT_DIR}/qwen3-4b-thaisum-adapter
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none

### train
per_device_train_batch_size: {PER_DEVICE_TRAIN_BATCH_SIZE}
gradient_accumulation_steps: {GRADIENT_ACCUMULATION_STEPS}
learning_rate: {LEARNING_RATE}
num_train_epochs: {NUM_TRAIN_EPOCHS}
lr_scheduler_type: {LR_SCHEDULER_TYPE}
warmup_ratio: {WARMUP_RATIO}
bf16: true
resume_from_checkpoint: null
"""

merge_yaml = f"""### model
model_name_or_path: {MODEL_NAME}
adapter_name_or_path: {OUTPUT_DIR}/qwen3-4b-thaisum-adapter
template: qwen3

### export
export_dir: {OUTPUT_DIR}/qwen3-4b-thaisum
export_size: 5
export_device: cpu
export_legacy_format: false
"""

Path(f'{YAML_DIR}/1_data-process-thaisum.yaml').write_text(preprocess_yaml, encoding='utf-8')
Path(f'{YAML_DIR}/2_qwen3_lora_sft_thaisum.yaml').write_text(train_yaml, encoding='utf-8')
Path(f'{YAML_DIR}/3_merge_lora_qwen3_thaisum.yaml').write_text(merge_yaml, encoding='utf-8')

print(preprocess_yaml)
print(train_yaml)
print(merge_yaml)

## 10. Preprocess / Tokenize with LLaMA Factory

Repo mapping: equivalent to the data-processing YAML flow in `script/yaml/1_data-process-thaisum.config.yaml`.

In [ ]:
!llamafactory-cli train {YAML_DIR}/1_data-process-thaisum.yaml

## 11. Train the LoRA Adapter

This is the main training step. On Colab, you may reduce `MAX_SAMPLES`, batch size, or epochs if runtime is tight.

In [ ]:
!llamafactory-cli train {YAML_DIR}/2_qwen3_lora_sft_thaisum.yaml

## 12. Merge Adapter into a Full Model

Repo mapping: equivalent to `script/yaml/3_merge_lora_qwen3_thaisum.config.yaml`.

In [ ]:
!llamafactory-cli export {YAML_DIR}/3_merge_lora_qwen3_thaisum.yaml

## 13. Load the Merged Model

In [ ]:
MERGED_MODEL_PATH = f'{OUTPUT_DIR}/qwen3-4b-thaisum'

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH,
    torch_dtype='auto',
    device_map='auto'
)

print('Loaded merged model from:', MERGED_MODEL_PATH)

## 14. Inference Helpers

Repo mapping: same style as `summarization_inference.py`.

In [ ]:
def postprocess_summary(text: str) -> str:
    text = text.strip()
    text = re.sub(r'^(?:สรุป|summary)\s*[:\-]\s*', '', text, flags=re.IGNORECASE)
    return text.strip()

@torch.inference_mode()
def summarize_text(body: str, max_new_tokens: int = MAX_NEW_TOKENS, temperature: float = 0.2) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': body + '\n/no_think'},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    model_inputs = tokenizer([prompt], return_tensors='pt').to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        repetition_penalty=1.1,
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
    output_text = tokenizer.decode(output_ids, skip_special_tokens=True)
    return postprocess_summary(output_text)

## 15. Manual Inference on Custom Thai Text

In [ ]:
custom_text = '''วางข่าวภาษาไทยหรือบทความภาษาไทยที่ต้องการสรุปที่นี่'''
print(summarize_text(custom_text))

## 16. Batch Inference on Validation Data

In [ ]:
validation_df = pd.read_json(f'{RAW_DATA_DIR}/validation.jsonl', lines=True)
validation_df = validation_df.head(EVAL_LIMIT).copy()
validation_df['generated_summary'] = validation_df['body'].apply(lambda x: summarize_text(str(x)))

prediction_path = f'{RAW_DATA_DIR}/validation_predictions.csv'
validation_df.to_csv(prediction_path, index=False)
validation_df[['body', 'summary', 'generated_summary']].head(5)

## 17. Evaluate with ROUGE

Repo mapping: same output idea as `evaluate_summarization.py`.

In [ ]:
from rouge_score import rouge_scorer

def compute_rouge(df, prediction_column='generated_summary', reference_column='summary'):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    totals = {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    valid_rows = 0

    for _, row in df.iterrows():
        prediction = str(row.get(prediction_column) or '').strip()
        reference = str(row.get(reference_column) or '').strip()
        if not prediction or not reference:
            continue

        scores = scorer.score(reference, prediction)
        for metric in totals:
            totals[metric] += scores[metric].fmeasure
        valid_rows += 1

    return {metric: round(total / valid_rows, 6) for metric, total in totals.items()}

metrics = compute_rouge(validation_df)
metrics

In [ ]:
metrics_path = f'{RAW_DATA_DIR}/validation_metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('Saved predictions to:', prediction_path)
print('Saved metrics to:', metrics_path)
print(json.dumps(metrics, ensure_ascii=False, indent=2))

## 18. What to Learn from This Notebook

Key ideas:
- ThaiSum provides `body -> summary` supervision for summarization SFT
- LLaMA Factory expects a dataset registry plus ShareGPT-formatted JSONL
- LoRA training saves a lightweight adapter first
- Merging creates a standalone model for easier inference
- ROUGE gives a fast automatic check, but you should also manually read generated summaries

How this maps back to the repo:
- If you want script-based execution, use the Python and YAML files in the repo root and `script/yaml/`
- If you want HPC execution, use the `submit-thaisum/` scripts
- If you only want interactive generation after training, compare with `notebook/manual_inference_thaisum.ipynb`